# DualSentinel — Pipeline Notebook

Simulates:
```bash
python src/pipeline.py \
  --input "data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv" \
  --dataset lmd \
  --skip-detectors \
  --threshold 0.0
```

Each cell corresponds to one pipeline step, making it easy to inspect intermediate results, re-run individual steps, and change parameters without re-running the entire pipeline.

**Requirements:** Ollama must be running (`ollama serve`) with `phi3:mini` and `llama3.2` pulled.

## Configuration
Edit these variables to change the run without touching any source files.

In [4]:
import sys, os
from pathlib import Path

# Locate the DualSentinel root regardless of where the kernel was launched from.
# The root is the first ancestor (or sibling) directory that owns src/pipeline.py.
_cwd = Path.cwd()
_candidates = [
    _cwd,                          # kernel started inside DualSentinel/
    _cwd / 'DualSentinel',         # kernel started at workspace root
    _cwd.parent,                   # kernel started inside DualSentinel/notebooks/
    _cwd.parent / 'DualSentinel',  # kernel started at Challange-3 parent
]
_root = next((p.resolve() for p in _candidates if (p / 'src' / 'pipeline.py').exists()), _cwd)
_src  = _root / 'src'

sys.path.insert(0, str(_src))
os.chdir(_root)   # ensures data/, results/ resolve correctly

import json, logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Parameters (mirrors the CLI flags) ─────────────────────────────────────
INPUT_PATH         = Path('data/samples/LMD-2023 [1.75M Elements - Normal]checked.csv')
DATASET            = 'lmd'
SKIP_DETECTORS     = True    # --skip-detectors
THRESHOLD          = 0.0     # --threshold 0.0
MAX_JUDGE_WINDOWS  = 50      # cap passed to judge_batch
WINDOW_SECONDS     = 60
MAX_EVENTS_PER_WINDOW = 200
SLM_MODEL   = os.getenv('SLM_MODEL',   'phi3:mini')
JUDGE_MODEL = os.getenv('JUDGE_MODEL', 'llama3.2')

# ── Output directory (one timestamped folder per run) ──────────────────────
RUN_TS     = datetime.now().strftime('%Y-%m-%d_%H-%M')
OUTPUT_DIR = Path('results') / RUN_TS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Working dir    : {Path.cwd()}')
print(f'src on path    : {_src}')
print(f'Input exists   : {INPUT_PATH.exists()}')
print(f'Output dir     : {OUTPUT_DIR}')
print(f'SLM model      : {SLM_MODEL}  |  Judge model: {JUDGE_MODEL}')
print(f'Threshold      : {THRESHOLD}  |  skip_detectors: {SKIP_DETECTORS}')


Working dir    : D:\ISEP\Challange-3\DualSentinel
src on path    : D:\ISEP\Challange-3\DualSentinel\src
Input exists   : True
Output dir     : results\2026-03-24_19-56
SLM model      : phi3:mini  |  Judge model: llama3.2
Threshold      : 0.0  |  skip_detectors: True


## Step 1 — Parse
Load the CSV, normalise column names, extract basenames from process paths, and filter to relevant Sysmon EventIDs.

In [5]:
from preprocessor import parse_csv

df = parse_csv(INPUT_PATH, dataset=DATASET)

print(f'Events loaded : {len(df):,}')
print(f'Time range    : {df.timestamp.min()}  →  {df.timestamp.max()}')
print(f'Unique EventIDs: {sorted(df.event_id.unique())}')
df.head()

INFO preprocessor: Loaded 687350 events from LMD-2023 [1.75M Elements - Normal]checked.csv (dataset=lmd)


Events loaded : 687,350
Time range    : 2026-03-24 00:00:06+00:00  →  2026-03-24 23:59:54+00:00
Unique EventIDs: [np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(8), np.int64(11), np.int64(12), np.int64(13), np.int64(15), np.int64(22)]


,unnamed: 0,label,name,guid,event_id,version,level,task,opcode,keywords,...,sourceprocessguid,targetprocessguid,newthreadid,startaddress,startmodule,startfunction,previouscreationutctime,hash,contents,id
0,25965,0,Microsoft-Windows-Sysmon,{5770385f-c22a-43e0-bf4c-06f5698ffbd9},22,5,4,22,0,0x8000000000000000,...,0,0,0.0,0,0,0,0,0,0,0
1,25964,0,Microsoft-Windows-Sysmon,{5770385f-c22a-43e0-bf4c-06f5698ffbd9},22,5,4,22,0,0x8000000000000000,...,0,0,0.0,0,0,0,0,0,0,0
2,7132,0,Microsoft-Windows-Sysmon,{5770385f-c22a-43e0-bf4c-06f5698ffbd9},1,3,4,1,0,0x8000000000000000,...,0,0,0.0,0,0,0,0,0,0,0
3,809689,0,Microsoft-Windows-Sysmon,{5770385f-c22a-43e0-bf4c-06f5698ffbd9},1,5,4,1,0,0x8000000000000000,...,0,0,0.0,0,0,0,0,0,0,0
4,809687,0,Microsoft-Windows-Sysmon,{5770385f-c22a-43e0-bf4c-06f5698ffbd9},1,5,4,1,0,0x8000000000000000,...,0,0,0.0,0,0,0,0,0,0,0


## Step 2 — Windowing & Feature Extraction
Slice events into 60-second tumbling windows and compute 17 numerical features per window, plus raw event summaries for the LLM evidence pack.

In [ ]:
from preprocessor import make_windows

windows = list(make_windows(df, window_size_seconds=WINDOW_SECONDS, max_events=MAX_EVENTS_PER_WINDOW))
wdf = pd.DataFrame([w.to_dict() for w in windows])
X = np.array([w.to_feature_vector() for w in windows])

print(f'Windows created : {len(windows)}')
print(f'Feature matrix  : {X.shape}')

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
wdf['event_count'].plot(ax=axes[0], title='Events per window', color='steelblue')
axes[0].set_xlabel('Window index')
wdf['network_connection_count'].plot(ax=axes[1], title='Network connections', color='coral')
axes[1].set_xlabel('Window index')
wdf[['powershell_count', 'cmd_count', 'suspicious_process_count']].plot(ax=axes[2], title='Suspicious process counts')
plt.tight_layout()
plt.show()

wdf[['event_count', 'unique_processes', 'network_connection_count',
     'suspicious_process_count', 'powershell_count', 'has_mimikatz', 'has_psexec',
     'event_id_entropy', 'process_entropy']].describe()

## Step 3 — IsolationForest  *(skipped: `--skip-detectors`)*
When `SKIP_DETECTORS=True` all `if_score` values are set to `0.0`. The cell below runs the detector anyway in **inspect mode** (results are not used for escalation) so you can see what it would have flagged.

In [ ]:
from detectors import IForestDetector

if_scores = np.zeros(len(windows))   # zeroed — SKIP_DETECTORS mode

# ── Inspect-only (not used for escalation) ─────────────────────────────────
iforest_inspect = IForestDetector()
iforest_inspect.fit(X)
if_scores_inspect = iforest_inspect.score(X)

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(len(if_scores_inspect)),
       if_scores_inspect,
       color=['red' if s > 0.6 else 'steelblue' for s in if_scores_inspect])
ax.axhline(0.6, color='orange', linestyle='--', label='Threshold 0.6')
ax.set_xlabel('Window index')
ax.set_ylabel('Anomaly score')
ax.set_title('IsolationForest scores (inspect only — not used for escalation in this run)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'[Inspect] Windows IF score > 0.6: {(if_scores_inspect > 0.6).sum()}/{len(windows)}')
print('NOTE: if_scores are zeroed for the actual pipeline run (SKIP_DETECTORS=True)')

## Step 4 — GRU Autoencoder  *(skipped: `--skip-detectors`)*
`gru_scores` are set to `0.0`. No model is trained in this mode.

In [ ]:
gru_scores = np.zeros(len(windows))   # zeroed — SKIP_DETECTORS mode
print('GRU skipped (SKIP_DETECTORS=True). gru_scores = all zeros.')

## Step 5 — ATT&CK Rule Tagger + Ensemble Score
The rule tagger always runs. With detectors skipped, `ensemble_score` is driven by ATT&CK rule hits only (max possible score = 0.2). All windows pass the `THRESHOLD=0.0` filter.

Ensemble weights: IF×0.5 + GRU×0.3 + rule_hit×0.2

In [ ]:
from detectors import tag_techniques, ensemble_score

window_dicts = []
for i, w in enumerate(windows):
    wd = w.to_dict()
    wd['attck_hits'] = tag_techniques(wd)
    wd['if_score']   = float(if_scores[i])
    wd['gru_score']  = float(gru_scores[i])
    wd['detector_score'] = ensemble_score(
        iforest_score=float(if_scores[i]),
        gru_score=float(gru_scores[i]),
        has_attck_hits=len(wd['attck_hits']) > 0,
    )
    window_dicts.append(wd)

wdf['detector_score'] = [w['detector_score'] for w in window_dicts]
wdf['attck_hit_count'] = [len(w['attck_hits']) for w in window_dicts]

# Save windows_scored.json
ws_path = OUTPUT_DIR / 'windows_scored.json'
with open(ws_path, 'w') as f:
    json.dump(window_dicts, f, indent=2, default=str)

high_risk = sum(1 for w in window_dicts if w['detector_score'] >= THRESHOLD)
print(f'Windows above threshold ({THRESHOLD}): {high_risk}/{len(window_dicts)}')
print(f'Saved → {ws_path}')

# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

score_values = wdf['detector_score']
axes[0].hist(score_values, bins=20, color='steelblue', edgecolor='white')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', label=f'Threshold={THRESHOLD}')
axes[0].set_xlabel('detector_score')
axes[0].set_ylabel('Windows')
axes[0].set_title('detector_score distribution')
axes[0].legend()

# ATT&CK techniques fired
from collections import Counter
tech_counts = Counter(
    h['name']
    for w in window_dicts for h in w['attck_hits']
)
if tech_counts:
    tc_s = pd.Series(tech_counts).sort_values()
    tc_s.plot(kind='barh', ax=axes[1], color='salmon', title='ATT&CK rule hits')
    axes[1].set_xlabel('Windows')
else:
    axes[1].text(0.5, 0.5, 'No ATT&CK rule hits', ha='center', va='center')
    axes[1].set_title('ATT&CK rule hits')

plt.tight_layout()
plt.show()

## Step 6a — SLM Analyst (`phi3:mini`)

Sends every window above `THRESHOLD` through the SLM for rapid first-pass triage. A trivially-benign pre-filter skips Ollama for windows with zero threat signals.

> **This cell calls Ollama.** Make sure `ollama serve` is running and `phi3:mini` is pulled.
> Interrupt the kernel to abort mid-run; already-collected results will not be saved.

In [ ]:
from slm_analyst import SLMAnalyst

analyst = SLMAnalyst(model=SLM_MODEL)
slm_analyses = analyst.analyse_batch(window_dicts, threshold=THRESHOLD)

slm_path = OUTPUT_DIR / 'slm_analyses.json'
with open(slm_path, 'w') as f:
    json.dump([a.to_dict() for a in slm_analyses], f, indent=2, default=str)

sdf = pd.DataFrame([a.to_dict() for a in slm_analyses])
prefiltered = sdf['summary'].str.contains('pre-filter', na=False).sum()

print(f'SLM analyses   : {len(sdf)}')
print(f'Pre-filtered   : {prefiltered}  (no Ollama call)')
print(f'Ollama calls   : {len(sdf) - prefiltered}')
print(f'Saved → {slm_path}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sdf['pre_score'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', title='pre_score distribution')
axes[0].set_xlabel('pre_score')
axes[0].set_ylabel('Windows')

color_map = {'low': 'green', 'medium': 'gold', 'high': 'orange', 'critical': 'red'}
rl = sdf['risk_level'].value_counts()
rl.plot(kind='bar', ax=axes[1],
        color=[color_map.get(l, 'grey') for l in rl.index],
        title='Risk level breakdown')
axes[1].set_xlabel('Risk level')
axes[1].set_ylabel('Windows')

sdf['needs_deep_analysis'].value_counts().plot(
    kind='bar', ax=axes[2], color=['green', 'orange'],
    title='needs_deep_analysis')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

cols = [c for c in ['window_start', 'pre_score', 'risk_level',
                     'needs_deep_analysis', 'summary'] if c in sdf.columns]
sdf[cols].sort_values('pre_score', ascending=False).head(20)

## Step 6b — LLM Judge (`llama3.2`)

Validates the SLM pre-diagnoses against the full evidence pack. Only the top `MAX_JUDGE_WINDOWS` (50) windows by `(detector_score, SLM pre_score)` are judged. Windows with `pre_score < 3` and `needs_deep_analysis=False` are skipped.

> **This cell calls Ollama.** Make sure `llama3.2` is pulled.

In [ ]:
from llm_judge import LLMJudge

judge = LLMJudge(model=JUDGE_MODEL)
judge_results = judge.judge_batch(
    window_dicts,
    slm_analyses=slm_analyses,
    threshold=THRESHOLD,
    max_windows=MAX_JUDGE_WINDOWS,
)

judge_path = OUTPUT_DIR / 'judge_results.json'
with open(judge_path, 'w') as f:
    json.dump([r.to_dict() for r in judge_results], f, indent=2, default=str)

print(f'Judge results: {len(judge_results)}')
print(f'Saved → {judge_path}')

if judge_results:
    jdf = pd.DataFrame([r.to_dict() for r in judge_results])

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    verdict_colors = {'normal': 'green', 'suspicious': 'orange', 'malicious': 'red'}
    vc = jdf['verdict'].value_counts()
    vc.plot(kind='bar', ax=axes[0],
            color=[verdict_colors.get(v, 'grey') for v in vc.index],
            title='Verdict distribution')
    axes[0].set_xlabel('Verdict')
    axes[0].set_ylabel('Count')

    axes[1].hist(jdf['anomaly_score'], bins=range(12), color='steelblue', edgecolor='white')
    axes[1].axvline(7, color='red', linestyle='--', label='High-risk threshold (≥7)')
    axes[1].set_xlabel('anomaly_score')
    axes[1].set_ylabel('Windows')
    axes[1].set_title('Anomaly score distribution')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    cols = [c for c in ['window_start', 'anomaly_score', 'verdict',
                         'slm_pre_score', 'detector_score', 'fp_risk'] if c in jdf.columns]
    jdf[cols].sort_values('anomaly_score', ascending=False)
else:
    print('No windows reached the Judge (all cleared by the SLM pre-filter or pre_score < 3).')

## Step 7 — Report
Generates the same Markdown report as the CLI pipeline.

In [ ]:
from pipeline import generate_report

report_path = generate_report(window_dicts, judge_results, OUTPUT_DIR, DATASET)
print(f'Report saved → {report_path}')

# Print a preview
text = report_path.read_text(encoding='utf-8')
print('\n' + '─' * 60)
print(text[:3000])
if len(text) > 3000:
    print(f'\n... (truncated, full report at {report_path})')

## Step 8 — Score Correlation
Cross-pipeline view: `detector_score` vs SLM `pre_score` vs Judge `anomaly_score`.

In [ ]:
if judge_results:
    _ws  = pd.DataFrame(window_dicts)[['window_start', 'detector_score']]
    _slm = sdf[['window_start', 'pre_score', 'risk_level']]
    _j   = jdf[['window_start', 'anomaly_score', 'verdict']]

    slm_merged   = _ws.merge(_slm, on='window_start', how='inner')
    judge_merged = _slm.merge(_j,  on='window_start', how='inner')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sc = axes[0].scatter(slm_merged['detector_score'], slm_merged['pre_score'],
                         alpha=0.4, s=20, c=slm_merged['pre_score'], cmap='RdYlGn_r')
    plt.colorbar(sc, ax=axes[0], label='pre_score')
    axes[0].set_xlabel('detector_score')
    axes[0].set_ylabel('SLM pre_score')
    axes[0].set_title('detector_score vs SLM pre_score')

    verdict_colors = {'normal': 'green', 'suspicious': 'orange', 'malicious': 'red'}
    for verdict, group in judge_merged.groupby('verdict'):
        axes[1].scatter(group['pre_score'], group['anomaly_score'],
                        label=verdict, color=verdict_colors.get(verdict, 'grey'),
                        alpha=0.7, s=60)
    axes[1].set_xlabel('SLM pre_score')
    axes[1].set_ylabel('Judge anomaly_score')
    axes[1].set_title('SLM pre_score vs Judge anomaly_score')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    r1 = slm_merged['detector_score'].corr(slm_merged['pre_score'])
    r2 = judge_merged['pre_score'].corr(judge_merged['anomaly_score'])
    print(f'Correlation (detector_score ↔ SLM pre_score)     : {r1:.3f}')
    print(f'Correlation (SLM pre_score  ↔ Judge anomaly_score): {r2:.3f}')
else:
    print('No judge results — run Step 6b first.')

## Run Summary

In [ ]:
malicious  = [r for r in judge_results if r.verdict == 'malicious']
suspicious = [r for r in judge_results if r.verdict == 'suspicious']
normal     = [r for r in judge_results if r.verdict == 'normal']

print('=' * 50)
print('PIPELINE RUN SUMMARY')
print('=' * 50)
print(f'Input          : {INPUT_PATH.name}')
print(f'Output dir     : {OUTPUT_DIR}')
print(f'Events parsed  : {len(df):,}')
print(f'Windows created: {len(windows):,}')
print(f'Threshold      : {THRESHOLD}  (skip_detectors={SKIP_DETECTORS})')
print(f'SLM calls      : {len(sdf) - prefiltered}  ({prefiltered} pre-filtered)')
print(f'Judge calls    : {len(judge_results)}')
print(f'  Malicious    : {len(malicious)}')
print(f'  Suspicious   : {len(suspicious)}')
print(f'  Normal       : {len(normal)}')
print(f'Report         : {report_path}')
print('=' * 50)